In [114]:
import pandas as pd

df = pd.read_csv("infra_cloud_data_01.csv")

# Step 1: Check null percentage
null_percent = df.isnull().mean() * 100

# Step 2: Drop columns with >90% nulls
cols_to_drop = null_percent[null_percent > 90].index
df_cleaned = df.drop(columns=cols_to_drop)

print("Dropped columns:", len(cols_to_drop))
print("Remaining columns:", df_cleaned.shape[1])

Dropped columns: 72
Remaining columns: 102


In [115]:
df_cleaned = df_cleaned.loc[:, ~df_cleaned.columns.str.contains(r'\.\d+$')]

In [116]:
import pandas as pd

df = pd.read_csv("infra_cloud_data_01.csv")

# Clean column names (important)
df.columns = df.columns.str.strip()

# Your selected columns
target_cols = [
    "Contact Full Name",
    "Title",
    "Department",
    "Seniority",
    "Company Name - Cleaned",
    "Website",
    "Contact LI Profile URL",
    "Contact City",
    "Contact State",
    "Contact Country",
    "Past Job",
    "Company Description"
]

# Keep only columns that exist
existing_cols = [col for col in target_cols if col in df.columns]

df_filtered = df[existing_cols]

print("Columns kept:", existing_cols)
print("Shape:", df_filtered.shape)



Columns kept: ['Contact Full Name', 'Title', 'Department', 'Seniority', 'Company Name - Cleaned', 'Website', 'Contact LI Profile URL', 'Contact City', 'Contact State', 'Contact Country', 'Past Job', 'Company Description']
Shape: (83, 12)


In [117]:
# Convert all "Total AI" columns to numeric
for i in range(1, 11):
    col = f"Email {i} Total AI"
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [118]:
def get_best_email(row):
    # 1. Primary email wins
    if pd.notnull(row.get("Primary Email")):
        return row["Primary Email"]
    
    best_email = None
    best_score = -1
    
    # 2. Check all 10 emails
    for i in range(1, 11):
        email_col = f"Email {i}"
        score_col = f"Email {i} Total AI"
        validation_col = f"Email {i} Validation"
        
        email = row.get(email_col)
        score = row.get(score_col)
        validation = str(row.get(validation_col)).lower()
        
        if pd.notnull(email):
            # Prefer validated emails
            is_valid = "valid" in validation or "verified" in validation
            
            if score is not None and score > best_score:
                if is_valid:
                    best_email = email
                    best_score = score
    
    return best_email

In [119]:
df["best_email"] = df.apply(get_best_email, axis=1)

In [120]:
df_final = df_filtered.copy()
df_final["best_email"] = df["best_email"]

In [121]:
# Drop critical missing rows
df_final = df_final.dropna(subset=["Contact Full Name", "Company Description"])

# Fill semantic fields
df_final["Title"] = df_final["Title"].fillna("Unknown Role")
df_final["Department"] = df_final["Department"].fillna("Unknown Department")
df_final["Past Job"] = df_final["Past Job"].fillna("")

# Fill metadata
df_final["Contact Country"] = df_final["Contact Country"].fillna("Unknown")
df_final["best_email"] = df_final["best_email"].fillna("NA")
df_final["Contact City"] = df_final["Contact City"].fillna("Unknown")
df_final["Contact State"] = df_final["Contact State"].fillna("Unknown")
df_final["Seniority"] = df_final["Seniority"].fillna("Unknown")

# Optional fields
df_final["Contact LI Profile URL"] = df_final["Contact LI Profile URL"].fillna("")
df_final["Website"] = df_final["Website"].fillna("")

In [122]:
df_final.to_csv("final_preprocessed_dataset.csv", index=False)

In [123]:
import pandas as pd

df = pd.read_csv("final_preprocessed_dataset.csv")

# Columns for RAG
rag_cols = [
    "Contact Full Name",
    "Title",
    "Department",
    "Company Name - Cleaned",
    "Company Description",
    "Past Job",
    "Contact Country",
    "Contact City",
    "Contact State",
    "Seniority"
]

# Keep only existing columns
rag_cols = [col for col in rag_cols if col in df.columns]

df_rag = df[rag_cols].copy()

# Save RAG dataset
df_rag.to_csv("rag_dataset.csv", index=False)

print("RAG dataset ready")

RAG dataset ready


In [124]:
def build_document(row):
    return f"""
    {row['Contact Full Name']} is a {row['Title']} in the {row['Department']} department 
    at {row['Company Name - Cleaned']}.

    The company operates as follows:
    {row['Company Description']}

    Previously worked as:
    {row.get('Past Job', '')}
    """

In [125]:
df_rag["document"] = df_rag.apply(build_document, axis=1)

In [126]:
df_rag = df_rag.drop_duplicates(
    subset=["Contact Full Name", "Company Name - Cleaned"]
)

In [127]:
pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [128]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
df_rag["embedding"] = df_rag["document"].apply(lambda x: model.encode(x))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2418.87it/s]


In [129]:
import psycopg2

conn = psycopg2.connect(
    dbname="semantic_search_db",
    user="postgres",
    password="disa",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

for _, row in df_rag.iterrows():
    cur.execute("""
        INSERT INTO contacts 
        (name, title, company, document, country, city, state, seniority, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row["Contact Full Name"],
        row["Title"],
        row["Company Name - Cleaned"],
        row["document"],
        row["Contact Country"],
        row["Contact City"],
        row["Contact State"],
        row["Seniority"],
        row["embedding"].tolist()
    ))

conn.commit()



In [139]:
query = "CTOs in fintech security India"
query_embedding = model.encode(query)
query_embedding = query_embedding.tolist()

In [140]:
query_embedding_str = "[" + ",".join(map(str, query_embedding)) + "]"

In [141]:
conn.rollback()

In [142]:
cur.execute("""
SELECT name, title, company, document
FROM contacts
WHERE country = 'India'
ORDER BY embedding <-> %s::vector
LIMIT 5;
""", (query_embedding_str,))

In [143]:
results = cur.fetchall()

for r in results:
    print("\n---")
    print("Name:", r[0])
    print("Title:", r[1])
    print("Company:", r[2])


---
Name: Vanamali R Sridharan
Title: Chief Technology Officer
Company: Five-Star Business Finance Limited

---
Name: Dinesh Maru
Title: Senior Vice President – IT Infrastructure & Information Security
Company: Neo Wealth and Asset Management

---
Name: Rafuzama Khan
Title: Deputy Vice President Information Security
Company: Shivalik Small Finance Bank

---
Name: Vikas Dhankhar
Title: Chief Technology Officer
Company: NeoGrowth Credit Pvt. Ltd.

---
Name: Pranesh Kumar Muppala
Title: Chief Technology Officer
Company: Olea
